# Ensemble: EfficientNet + Swin-B

In [1]:
import os
import re
import random
import numpy as np
import torch
import torch.nn as nn
import pandas as pd
from tqdm.auto import tqdm
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Dataset
from PIL import Image

base_dir   = '/workspace/144FinalProjectDataset'
TRAIN_PATH = os.path.join(base_dir, 'train')
TEST_PATH  = os.path.join(base_dir, 'test')
BATCHSIZE  = 32
NUM_WORKERS = 10
PREFETCH_FACTOR = 3

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print('device:', device)

device: cuda


## Configuration

In [2]:
EFFNET_CKPT_PATH  = './checkpoints/efficientnet_best.pth'
SWINB_CKPT_PATH   = './checkpoints/swinb_best.pth'
NUM_TTA_PASSES    = 5
EFFNET_IMAGE_SIZE = 384
SWINB_IMAGE_SIZE  = 384

In [3]:
missing = []
if not os.path.exists(EFFNET_CKPT_PATH):
    missing.append(f'EfficientNet checkpoint: {EFFNET_CKPT_PATH}')
if not os.path.exists(SWINB_CKPT_PATH):
    missing.append(f'Swin-B checkpoint:       {SWINB_CKPT_PATH}')

if missing:
    raise FileNotFoundError('Missing checkpoints:\n' + '\n'.join(f'  - {m}' for m in missing))

print('Both checkpoints found.')
print(f'  EfficientNet: {EFFNET_CKPT_PATH}')
print(f'  Swin-B:       {SWINB_CKPT_PATH}')

Both checkpoints found.
  EfficientNet: ./checkpoints/efficientnet_best.pth
  Swin-B:       ./checkpoints/swinb_best.pth


In [4]:
class TestDataset(Dataset):
    def __init__(self, test_dir, transform=None):
        self.test_dir  = test_dir
        self.transform = transform

        jpg_files = [f for f in os.listdir(test_dir) if f.endswith('.jpg')]

        def sort_key(fname):
            m = re.findall(r'(\d+)\.jpg', fname)
            return int(m[0]) if m else float('inf')

        self.image_names = sorted(jpg_files, key=sort_key)
        print(f'TestDataset: {len(self.image_names)} images found in {test_dir}')

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        image_name = self.image_names[idx]
        image = Image.open(os.path.join(self.test_dir, image_name)).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, image_name

In [5]:
# Build label mapping: use the alphabetical inverse of ImageFolder's class_to_idx,
# which matches how the models were actually trained (DataLoader returns alphabetical
# targets via Subset.__getitem__ → underlying ImageFolder, regardless of any target
# remapping applied to the Subset object).
full_train_dataset   = datasets.ImageFolder(root=TRAIN_PATH)
idx_to_correct_label = {v: int(k) for k, v in full_train_dataset.class_to_idx.items()}
print('Label mapping built. Sample:', {k: idx_to_correct_label[k] for k in range(5)})

Label mapping built. Sample: {0: 0, 1: 1, 2: 10, 3: 11, 4: 12}


## EfficientNet TTA Inference

In [6]:
effnet_tta_transform = transforms.Compose([
    transforms.Resize((EFFNET_IMAGE_SIZE, EFFNET_IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=5),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

effnet = models.efficientnet_v2_s(weights=None)
in_features = effnet.classifier[1].in_features
effnet.classifier = nn.Sequential(nn.Dropout(0.4), nn.Linear(in_features, 100))

state_dict = torch.load(EFFNET_CKPT_PATH, map_location=device)['model_state_dict']
state_dict = {k.replace('_orig_mod.', ''): v for k, v in state_dict.items()}
effnet.load_state_dict(state_dict)

effnet = effnet.to(device).eval()
print('EfficientNet loaded.')

effnet_test_dataset = TestDataset(TEST_PATH, transform=effnet_tta_transform)
effnet_test_loader  = DataLoader(effnet_test_dataset, batch_size=BATCHSIZE, shuffle=False, num_workers=NUM_WORKERS, persistent_workers=True, prefetch_factor=PREFETCH_FACTOR)

image_ids    = None
effnet_probs = None

with torch.no_grad():
    for pass_idx in range(NUM_TTA_PASSES):
        pass_probs = []
        pass_ids   = []
        for images, names in tqdm(effnet_test_loader, desc=f'EfficientNet TTA pass {pass_idx+1}/{NUM_TTA_PASSES}'):
            images = images.to(device)
            probs  = torch.softmax(effnet(images), dim=1)
            pass_probs.append(probs.cpu())
            if pass_idx == 0:
                pass_ids.extend(names)
        pass_probs   = torch.cat(pass_probs, dim=0)
        effnet_probs = pass_probs if effnet_probs is None else effnet_probs + pass_probs
        if pass_idx == 0:
            image_ids = pass_ids

effnet_probs /= NUM_TTA_PASSES
print(f'EfficientNet probs shape: {effnet_probs.shape}')

EfficientNet loaded.
TestDataset: 1036 images found in /workspace/144FinalProjectDataset/test


EfficientNet TTA pass 1/5:   0%|          | 0/33 [00:00<?, ?it/s]

EfficientNet TTA pass 2/5:   0%|          | 0/33 [00:00<?, ?it/s]

EfficientNet TTA pass 3/5:   0%|          | 0/33 [00:00<?, ?it/s]

EfficientNet TTA pass 4/5:   0%|          | 0/33 [00:00<?, ?it/s]

EfficientNet TTA pass 5/5:   0%|          | 0/33 [00:00<?, ?it/s]

EfficientNet probs shape: torch.Size([1036, 100])


## Swin-B TTA Inference

In [7]:
swin_tta_transform = transforms.Compose([
    transforms.Resize((SWINB_IMAGE_SIZE, SWINB_IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=5),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

swin = models.swin_b(weights=None)
in_features = swin.head.in_features
swin.head = nn.Sequential(nn.Dropout(0.4), nn.Linear(in_features, 100))

state_dict = torch.load(SWINB_CKPT_PATH, map_location=device)['model_state_dict']
state_dict = {k.replace('_orig_mod.', ''): v for k, v in state_dict.items()}
swin.load_state_dict(state_dict)

swin = swin.to(device).eval()
print('Swin-B loaded.')

swin_test_dataset = TestDataset(TEST_PATH, transform=swin_tta_transform)
swin_test_loader  = DataLoader(swin_test_dataset, batch_size=BATCHSIZE, shuffle=False, num_workers=NUM_WORKERS, persistent_workers=True, prefetch_factor=PREFETCH_FACTOR)

swin_probs = None

with torch.no_grad():
    for pass_idx in range(NUM_TTA_PASSES):
        pass_probs = []
        for images, _ in tqdm(swin_test_loader, desc=f'Swin-B TTA pass {pass_idx+1}/{NUM_TTA_PASSES}'):
            images = images.to(device)
            probs  = torch.softmax(swin(images), dim=1)
            pass_probs.append(probs.cpu())
        pass_probs = torch.cat(pass_probs, dim=0)
        swin_probs = pass_probs if swin_probs is None else swin_probs + pass_probs

swin_probs /= NUM_TTA_PASSES
print(f'Swin-B probs shape: {swin_probs.shape}')

Swin-B loaded.
TestDataset: 1036 images found in /workspace/144FinalProjectDataset/test


Swin-B TTA pass 1/5:   0%|          | 0/33 [00:00<?, ?it/s]

Swin-B TTA pass 2/5:   0%|          | 0/33 [00:00<?, ?it/s]

Swin-B TTA pass 3/5:   0%|          | 0/33 [00:00<?, ?it/s]

Swin-B TTA pass 4/5:   0%|          | 0/33 [00:00<?, ?it/s]

Swin-B TTA pass 5/5:   0%|          | 0/33 [00:00<?, ?it/s]

Swin-B probs shape: torch.Size([1036, 100])


## Ensemble & Generate Submission

In [8]:
avg_probs = (effnet_probs + swin_probs) / 2 # Weight Both Equally
_, preds  = torch.max(avg_probs, 1)

labels = [idx_to_correct_label[p] for p in preds.numpy()]

df = pd.DataFrame({'ID': image_ids, 'Label': labels})
df['ID_num'] = df['ID'].apply(lambda x: int(os.path.splitext(x)[0]))
df = df.sort_values('ID_num').drop(columns=['ID_num']).reset_index(drop=True)
df.to_csv('submission_ensemble.csv', index=False)

print('submission_ensemble.csv created!')
print(df.head())

submission_ensemble.csv created!
      ID  Label
0  0.jpg     62
1  1.jpg     43
2  2.jpg     38
3  3.jpg     51
4  4.jpg     42
